# SysML v2 Python Units: Documentation

This notebook shows how to use the `sysmlv2_units` library.
For a general intro and installation instructions, refer to the [main readme](README.md).

This documentation assumes familiarity with [SysML v2](https://sensmetry.com/advent-of-sysml-v2/),
[Syside Automator](https://docs.sensmetry.com/automator/) and [Pint](https://pint.readthedocs.io/).

In [1]:
import syside
import tempfile

# Helper function for loading SysML v2 text as a Syside model
def read_sysml(sysml_text: str):
    with tempfile.TemporaryDirectory() as tmp_dir:
        # Write SysML v2 textual notation to temporary file
        model_file = f'{tmp_dir}/model.sysml'
        with open(model_file, 'wb') as fp:
            fp.write(sysml_text.encode('utf-8'))
        
        # Load the model using Syside, raise if there are any errors or warnings
        try:
            model, _ = syside.load_model([model_file], warnings_as_errors=True)
        except syside.ModelError as e:
            for diag in e.diagnostics.all:
                print(str(diag))
            raise
        
    # Get the root elements
    doc_ctx = model.documents[0].lock()
    doc = doc_ctx.__enter__()
    
    root = doc.root_node
    elements = list(root.children.elements)
    return model, doc, elements

# Helper function for creating a new empty Syside model
def create_empty_sysml_model():
    model, _ = syside.load_model([])
    model.documents.append(syside.Document.create_st(
        syside.DocumentOptions(url=syside.Url('memory://new'), language='sysml'),
    ))
    
    doc_ctx = model.documents[0].lock()
    doc = doc_ctx.__enter__()
    return model, doc

## Quick Example

In [2]:
import math
import syside
from sysmlv2_units import SysMLUnitsHelper, ureg, UndefinedUnitError

# Read SysML v2 textual notation using Syside Automator
model, doc, elements = read_sysml("""
private import SI::*;
package ThePackage {
    part def PartWithAttributes {
    
        // Simple units
        attribute length = 1000 [m];
        attribute mass = 5000 [kg];
        
        // Unit prefixes
        attribute energy = 2.5 [MJ];
        attribute diameter = 3.14 [cm];
        
        // Compound units
        attribute speed = 300 [m/s];
        attribute pressure = 101325 [N/m^2]; // Normally you would use [Pa]
        
        // Compound unit defined in standard lib
        attribute acceleration = 10.0 ['m⋅s⁻²'];

        // Compound unit not defined in the standard lib
        attribute crazyUnits = -3.14 [K*kg*s**2/m^3];
        
        // Temperature
        attribute absTemp = 15.0 ['°C_abs']; // = 288.15 K
        attribute deltaTemp = 15.0 ['°C']; // = 15.0 K
        
        // Attribute with a quantity base type
        attribute density :> ISQ::massDensity;

    }
}
""")

# Create units helper
units_helper = SysMLUnitsHelper(model)

# Get the attributes of the part
part = elements[1].children.elements[0]
assert part.name == 'PartWithAttributes'
attrs = list(part.children.elements)

In [3]:
# Read attribute value quantity
q = units_helper.get_quantity(attrs[0])
assert q.magnitude == 1000
assert str(q.units) == 'meter'

# Only read the unit
units, m_unit_attr = units_helper.get_units(attrs[0])
assert units == ureg.m
assert str(units) == 'meter'

# SysML v2 units --> Pint units
assert isinstance(m_unit_attr, syside.AttributeUsage)
assert m_unit_attr.owner.name == 'SI'
assert units_helper.get_python_units(m_unit_attr) == ureg.m

# Pint units --> SysML v2 units
assert units_helper.get_sysml_units(ureg.m) == m_unit_attr
assert units_helper.get_sysml_units('m') == m_unit_attr

# Pint parsing helpers
assert units_helper.parse_python_units('m/s') == ureg['m/s']

In [4]:
# Read and print all attribute values
def _read_print_quantity(feature: syside.Feature):
    # ~P formats the units using the short form and with unicode support
    # https://pint.readthedocs.io/en/stable/getting/tutorial.html#string-formatting
    try:
        print(f'  {feature.name} = {units_helper.get_quantity(feature):~P}')
    except ValueError:
        print(f'  {feature.name} has no value')
    
print('Attribute quantities:')
for attr in attrs:
    _read_print_quantity(attr)
print('')

Attribute quantities:
  length = 1000 m
  mass = 5000 kg
  energy = 2.5 MJ
  diameter = 3.14 cm
  speed = 300 m/s
  pressure = 101325 N/m²
  acceleration = 10.0 m/s²
  crazyUnits = -3.14 K·kg·s²/m³
  absTemp = 15.0 °C
  deltaTemp = 15.0 Δ°C
  density has no value



In [5]:
# Convert value to another unit (m --> km)
q_m = units_helper.get_quantity(attrs[0])
q_km = q_m.to(ureg.km)
units_helper.set_quantity(attrs[0], q_km)

# kg --> lbs
units_helper.set_quantity(
    attrs[1], units_helper.get_quantity(
        attrs[1]).to(ureg.lbs))

# Temperature conversion
units_helper.set_quantity(
    attrs[8], units_helper.get_quantity(attrs[8]).to(ureg.K))
units_helper.set_quantity(
    attrs[9], units_helper.get_quantity(attrs[9]).to(ureg.K))

# Set quantity: quantity constructor
units_helper.set_quantity(
    attrs[2], units_helper.quantity(150, ureg.kJ))

units_helper.set_quantity(attrs[3], 3.14 * ureg.m)

# Set quantity: value and parsed unit str
units_helper.set_value_and_units(attrs[4], 42, 'km/h')

# Set special values: inf and NaN
units_helper.set_value_and_units(attrs[5], math.inf, 'Pa')
units_helper.set_value_and_units(attrs[6], math.nan, 'm/s**2')

# Get and set compound units
quantity = units_helper.get_quantity(attrs[7])
assert len(list(quantity.units._units.unit_items())) > 1
units_helper.set_quantity(attrs[7], quantity*2)

# Get preferred units of a quantity base type
quantity_attr = attrs[10]
assert units_helper.is_typed_by_quantity_value(quantity_attr)
units = units_helper.get_quantity_value_units(quantity_attr)
assert units == ureg['kg/m**3']

units, _ = units_helper.get_units(quantity_attr)
assert units == ureg['kg/m**3']

units_helper.set_quantity(quantity_attr, 1.225 * units)
assert units_helper.get_quantity(quantity_attr) == 1.225 * units

# Read and print all attribute values
print('Modified attribute quantities:')
for attr in attrs:
    _read_print_quantity(attr)
print('')

Modified attribute quantities:
  length = 1.0 km
  mass = 11023.113109243879 lb
  energy = 150 kJ
  diameter = 3.14 m
  speed = 42 kph
  pressure = inf Pa
  acceleration = nan m/s²
  crazyUnits = -6.28 K·kg·s²/m³
  absTemp = 288.15 K
  deltaTemp = 15.0 K
  density = 1.225 kg/m³



In [6]:
# Pretty-print to SysML v2 textual notation
print(units_helper.to_text(doc.root_node))

private import SI::*;
package ThePackage {
    part def PartWithAttributes {
        attribute length = 1 [km];
        attribute mass = 11023.113109243879 [USCU::lb];

        attribute energy = 150 [kJ];
        attribute diameter = 3.14 [m];

        attribute speed = 42 ['km/h'];
        attribute pressure = * [Pa];

        attribute acceleration = null ['m⋅s⁻²'];

        attribute crazyUnits = -6.28 [K * kg * s^2 * m^(-3)];

        attribute absTemp = 288.15 [K];
        attribute deltaTemp = 15 [K];

        attribute density :> ISQ::massDensity = 1.225 ['kg⋅m⁻³'];
    }
} // package ThePackage



## Getting and Setting Quantities

Quantities are a combination of a value (int, float, `inf`, `NaN`) and units.
Quantities are specified using Pint's `Quantity` object, which you can
[create in various ways](https://pint.readthedocs.io/en/stable/getting/tutorial.html#defining-a-quantity).

In [7]:
# Create empty model with an attribute
model, doc = create_empty_sysml_model()

_, attr = doc.root_node.children.append(syside.FeatureMembership, syside.AttributeUsage)
attr.declared_name = 'myAttribute'

# Instantiate units helper
units_helper = SysMLUnitsHelper(model)

You can set the value to some quantity:

In [8]:
quantity = 3.14 * ureg.m
print(quantity)

units_helper.set_quantity(attr, quantity)

print(units_helper.to_text(attr))

3.14 meter
attribute myAttribute = 3.14 [SI::m];



You can also set the value and units separately, which can make it easier to parse the units from a string:

In [9]:
units_helper.set_value_and_units(attr, 3.14, 'm/s^2')
print(units_helper.to_text(attr))

attribute myAttribute = 3.14 [SI::'m⋅s⁻²'];



You can get the quantity that was set:

In [10]:
quantity = units_helper.get_quantity(attr)
assert quantity.magnitude == 3.14
assert quantity.units == ureg['m/s^2']
print(quantity)
print(f'{quantity:~P}')  # Shorthand pretty-printing

3.14 meter / second ** 2
3.14 m/s²


Since we are using Pint we can use all its features, for example unit conversion:

In [11]:
quantity_si = units_helper.get_quantity(attr)
quantity_uscu = quantity_si.to(ureg['ft/s^2'])
print(quantity_uscu)

units_helper.set_quantity(attr, quantity_uscu)
print(units_helper.to_text(attr))

10.301837270341208 foot / second ** 2
attribute myAttribute = 10.301837270341208 [USCU::'ft/s²'];



Getting and setting dimensionless quantities:

In [12]:
units_helper.set_value_and_units(attr, 42)
print(units_helper.to_text(attr))

quantity = units_helper.get_quantity(attr)
print(quantity)
print(f'{quantity:~P}')

attribute myAttribute = 42;

42 dimensionless
42


Setting a quantity with unknown units raises an `UndefinedUnitError`:

In [13]:
# Undefined Pint units
try:
    units_helper.set_value_and_units(attr, 666, 'blabla')
except UndefinedUnitError as e:
    print(str(e))

# Undefined SysML units
print(ureg['cal'])
try:
    units_helper.set_value_and_units(attr, 666, 'cal')
except UndefinedUnitError as e:
    print(str(e))

'blabla' is not defined in the unit registry
1 calorie
Cannot map Pint units to SysML: calorie


Since quantity setting is done on a feature, you can also set the quantity on a feature that is part of a more complex expression:

In [14]:
# Create an operator expression with the "plus" operator as the value of our attribute
_, expression = attr.feature_value_member.set_member_element(syside.OperatorExpression)
expression.operator = syside.ExplicitOperator.Plus

# Create other attribute and set a quantity as its value
_, other_attr = attr.owner.children.insert(0, syside.FeatureMembership, syside.AttributeUsage)
other_attr.declared_name = 'otherAttribute'
units_helper.set_quantity(other_attr, 3140 * ureg.W)

# Create the left-side parameter and set it as a reference to the other attribute
_, left_param_feature = expression.children.append(syside.ParameterMembership, syside.Feature)
_, ref_expression = left_param_feature.feature_value_member.set_member_element(syside.FeatureReferenceExpression)
ref_expression.referent_member.set_member_element(other_attr)

# Create the right-side parameter and set a quantity as its value
_, right_param_feature = expression.children.append(syside.ParameterMembership, syside.Feature)
units_helper.set_quantity(right_param_feature, 42 * ureg.kW)

print(units_helper.to_text(attr.owner))

attribute otherAttribute = 3140 [SI::W];
attribute myAttribute = otherAttribute + 42 [SI::kW];



Getting and setting compound units works.
A custom units expression is built if the compound units are not defined already in the standard library.

In [15]:
# Compound units also available as a separate units attribute: m/s
model, _, elements = read_sysml("attribute speed = 42 [SI::m / SI::s];")
units_helper = SysMLUnitsHelper(model)

attr = elements[0]
quantity = units_helper.get_quantity(attr)
print(quantity)

units_helper.set_quantity(attr, quantity)
print(units_helper.to_text(attr))

# Compound units not available as a separate units attribute: K*kg/m**3
model, _, elements = read_sysml("attribute valueWithCustomUnits = 42 [SI::K * SI::kg / SI::m^3];")
units_helper = SysMLUnitsHelper(model)

attr = elements[0]
quantity = units_helper.get_quantity(attr)
print(quantity)

units_helper.set_quantity(attr, quantity)
print(units_helper.to_text(attr))

42 meter / second
attribute speed = 42 [SI::'m/s'];

42 kelvin * kilogram / meter ** 3
attribute valueWithCustomUnits = 42 [SI::K * SI::kg * SI::m^(-3)];



## Getting and Setting Units

You can also use the library get and set units as values.
This can be helpful if you need to explicitly store the units you have been using in some context, or if you need lower-level access to the quantities of your attributes.

In [16]:
# Create empty model with an attribute
model, doc = create_empty_sysml_model()

_, attr = doc.root_node.children.append(syside.FeatureMembership, syside.AttributeUsage)
attr.declared_name = 'myAttribute'

# Instantiate units helper
units_helper = SysMLUnitsHelper(model)

Set a quantity and get the Pint units and (if possible) the units attribute of the standard library:

In [17]:
units_helper.set_quantity(attr, 2.5 * ureg.km)
print(units_helper.to_text(attr))

units, units_attr = units_helper.get_units(attr)
assert units == ureg.km
print(units_attr)

attribute myAttribute = 2.5 [SI::km];

SI::kilometre


Get and set the units as the value itself:

In [18]:
units_helper.set_units(attr, 'km')
print(units_helper.to_text(attr))

units, units_attr = units_helper.get_units(attr)
assert units == ureg.km
assert str(units_attr) == 'SI::kilometre'

attribute myAttribute = SI::km;



This also works with compound units:

In [19]:
units_helper.set_units(attr, 'K*kg/m')
print(units_helper.to_text(attr))

units, units_attr = units_helper.get_units(attr)
assert units == ureg('K*kg/m').units
assert units_attr is None  # There is no single units attribute that represents these compound units

attribute myAttribute = SI::K * SI::kg * SI::m^(-1);



Get preferred units derived from a base quantity type:

In [20]:
model, _, elements = read_sysml("attribute mySpeed :> ISQ::speed;")
units_helper = SysMLUnitsHelper(model)

attr = elements[0]
units, _ = units_helper.get_units(attr)
assert units == ureg['m/s']

## Converting Between SysML v2, Pint and str

### Convert between string and Pint units

In [21]:
units = units_helper.parse_python_units('m/s**2')
assert units == ureg['m/s^2']

units_str = str(units)
units_str_pretty = f'{units:~P}'
units_str_sysml_style = f'{units:~S}'

print(units_str)
print(units_str_pretty)
print(units_str_sysml_style)

assert units_helper.parse_python_units(units_str) == ureg['m/s^2']
assert units_helper.parse_python_units(units_str_pretty) == ureg['m/s^2']

meter / second ** 2
m/s²
m⋅s⁻²


### Convert between Pint or str units and SysML units

Note: this only works for non-compound units (working with SysML units expressions is only implemented for getting/setting units/quantities as feature values)!

In [22]:
units_attr = units_helper.get_sysml_units('km')
assert str(units_attr) == 'SI::kilometre'

units = units_helper.get_python_units(units_attr)
assert units == ureg.km

# Trying to convert compound units results in an exception
try:
    units_helper.get_sysml_units('K*kg/m')
except UndefinedUnitError as e:
    print(str(e))

Cannot map Pint units to SysML: kelvin * kilogram / meter


## Printing SysML v2 Models

Syside can output SysML v2 using [pprint](https://docs.sensmetry.com/python/latest/syside/index.html#syside.pprint).
By default, this would print references to units attributes (of the standard lib) using their long names,
which is usually not how people read units.

For example:

In [23]:
# Create empty model with attributes
model, doc = create_empty_sysml_model()

_, attr = doc.root_node.children.append(syside.FeatureMembership, syside.AttributeUsage)
attr.declared_name = 'quantityAttr'
_, attr2 = doc.root_node.children.append(syside.FeatureMembership, syside.AttributeUsage)
attr2.declared_name = 'compoundUnitsAttr'

# Instantiate units helper
units_helper = SysMLUnitsHelper(model)

# Set quantities to the attributes
units_helper.set_quantity(attr, 3.14 * ureg['m/s**2'])
units_helper.set_quantity(attr2, ureg('42 K*kg/m'))

# Print the model
printer = syside.ModelPrinter.sysml()
printer_config = syside.PrinterConfig(line_width=120, tab_width=4)
print(syside.pprint(doc.root_node, printer, printer_config))

attribute quantityAttr = 3.14 [SI::'metre second to the power minus 2'];
attribute compoundUnitsAttr = 42 [SI::kelvin * SI::kilogram * SI::metre^(-1)];



To make this more readable, the library provides a `ReferencePrinter` which ensures that units are printed using their short name:

In [24]:
reference_printer = units_helper.get_reference_printer()
printer = syside.ModelPrinter.sysml(reference_printer=reference_printer)
print(syside.pprint(doc.root_node, printer, printer_config))

attribute quantityAttr = 3.14 [SI::'m⋅s⁻²'];
attribute compoundUnitsAttr = 42 [SI::K * SI::kg * SI::m^(-1)];



You can also simply use the `UnitsHelper.to_text` function so that you don't need to configure the printer:

In [25]:
print(units_helper.to_text(doc.root_node))

attribute quantityAttr = 3.14 [SI::'m⋅s⁻²'];
attribute compoundUnitsAttr = 42 [SI::K * SI::kg * SI::m^(-1)];

